In [17]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()

In [18]:
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [ ]:
# 데이터 분할(8:2)
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

train_set, test_set = train_test_split(housing,test_size=0.2,random_state=42)
# 훈련용데이터 x y(median_house_value)를 분리
housing_train = train_set.drop('median_house_value',axis=1)
housing_labels = train_set['median_house_value'].copy()
#  수치형 , 범주형 열 파악
num_attribs = list(housing_train.select_dtypes(np.number).columns)
cat_attribs = ['ocean_proximity']

# 파이프라인 구축
num_pipeline = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())
])
# 범주형 열과 수치형 열 처리를 하나로 묶어 ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
preprocessing =  ColumnTransformer([
    ('num',num_pipeline,num_attribs),
    ('cat',OneHotEncoder(),cat_attribs)
])

# 선형회귀 모델
from sklearn.linear_model import LinearRegression
pipeline = Pipeline([
    ('preprocess',preprocessing),('model',LinearRegression())
])

#학습
pipeline.fit(housing_train,housing_labels)
#평가
housing_test = test_set.drop('median_house_value',axis=1)
housing_test_labels = test_set['median_house_value'].copy()

pipeline.score(housing_test, housing_test_labels)

0.6486352079787185

In [23]:
# 의사결정나무(Decition Tree Regressor)
#- CART(Classification and Regression Tree) 알고리즘 
from sklearn.tree import DecisionTreeRegressor
tree_pipeline = Pipeline([
    ('preprocess',preprocessing),('model',DecisionTreeRegressor())
])
tree_pipeline.fit(housing_train,housing_labels)
tree_pipeline.score(housing_test,housing_test_labels)

0.66309882114316

In [24]:
#랜덤 포레스트
from sklearn.ensemble import RandomForestRegressor
random_pipeline = Pipeline([
    ('preprocess',preprocessing),('model',RandomForestRegressor())
])
random_pipeline.fit(housing_train,housing_labels)
random_pipeline.score(housing_test,housing_test_labels)

0.8229246944401302

In [43]:
# 원본에서 랜덤하게 5개 데이터 추출
import random
random.seed(42)
random_5 = random.sample(range(len(housing)), 5)
print(random_5)
print( random_pipeline.predict(housing.iloc[random_5].drop('median_house_value',axis=1)) )
print( housing.iloc[random_5]['median_house_value'].values)

[3648, 819, 9012, 8024, 7314]
[ 82530.    86107.   269060.   130180.   347419.01]
[ 77200. 100000. 275000. 136500. 327300.]


In [ ]:
# 튜닝 - 모델